In [294]:
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
import time
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import numpy as np

import random

def set_seed(seed):
    """Sets the seed for reproducibility."""
    torch.manual_seed(seed)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # if using multi-GPU.
        
    np.random.seed(seed)  # Numpy module.
    random.seed(seed)  # Python's random module.
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.enabled = False

set_seed(1000)

In [87]:
# Import pre-split datasets
df_train = pd.read_csv('./hotdogs-train.csv')
df_test = pd.read_csv('./hotdogs-test.csv')

## Text processing

In [303]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def preprocess_text(text):
    # Some reviews are only numbers (Why? I don't know). Review should always be treated as a string
    text = str(text)
    
    # This appears in some reviews
    text = text.replace('(Translated by Google)', ' ')
    text = text.replace('\n', ' ')

    # Convert to lowercase
    text = text.lower()
    
    # Remove non letter or number entries
    # BERT does OK with numbers is seems
    text = re.sub(r'[^\w\s]', '', text)
    
    return text

# Process the text for better classification
df_train['text'] = df_train['text'].apply(preprocess_text)
df_test['text'] = df_test['text'].apply(preprocess_text)

In [91]:
# Turn ratings into +/-/neutral and do a bit of light processing to the text
# Careful to only run this once

def pnn(n):
    if n in {1, 2}: return 0
    elif n in {3}: return 1
    else: return 2

def minus1(n):
    return n-1
    
df_train['pnn'] = df_train['rating'].apply(pnn)
df_test['pnn'] = df_test['rating'].apply(pnn)


# df_train['rating'] = df_train['rating'].apply(minus1)
# df_test['rating'] = df_test['rating'].apply(minus1)

In [93]:
print('Train set sample')
print(df_train.sample(8))
print()

print('Test set sample')
print(df_test.sample(8))

Train set sample
       rating                                               text  pnn
45215       3                      good food always cooked right    2
58665       1  the beef was yuck onion rings burnt and no pep...    0
34588       4  great food the fries and hotdogs are excellent...    2
48423       3                                     chocolate cake    2
568         4                               good food long lines    2
23371       4  fast service crew member took order and we pai...    2
4766        4       good beef and dogs cheese fries are the best    2
7544        4  u already know no matter what u pick from here...    2

Test set sample
       rating                                               text  pnn
5013        4  some of the best chicagosyle fast food outside...    2
4272        4                                         great food    2
426         4  big dip is the best on everything the most ama...    2
13489       2             its a hot dog place what do yo

Instantiate the bert model. Tried both base and large, no real difference in acc

## Create BERT instance

In [268]:
from transformers import BertTokenizer, BertModel, BertForSequenceClassification
from transformers import AutoModelForSequenceClassification, TFAutoModelForSequenceClassification
from transformers import AutoTokenizer, AutoConfig, get_linear_schedule_with_warmup

model_name = f"cardiffnlp/twitter-roberta-base-sentiment-latest"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
config = AutoConfig.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)

# Model stats
#print(model)
print('Device: ', device)

/opt/anaconda3/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactl

Device:  cpu


## Test the output of an untrained model

In [292]:
# Simple test cycle of the untrained model
def predict_sentiment(text):
    inputs = tokenizer(text, padding="max_length", truncation=True, max_length=128, return_tensors="pt")
    
    input_ids = inputs['input_ids']
    attention_mask = inputs['attention_mask']

    inputs = {'input_ids':input_ids, 'attention_mask':attention_mask}
    
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1)
        predicted_class = torch.argmax(probabilities, dim=1).item()
        
    return predicted_class, probabilities.tolist()

# Enter your text here
review = 'The table was dirty but the food was decent'
label, probs = predict_sentiment(review)
meaning = ['negative', 'neutral/average', 'good']

print(f'Review: {review}')
print(f'Predicted rating: {label} ({meaning[label]})')
print(f'Probabilities: 0 ({probs[0][0]*100:.2f}%) | 1 ({probs[0][1]*100:.2f}%) | 2 ({probs[0][2]*100:.2f}%)')

Review: The table was dirty but the food was decent
Predicted rating: 2 (good)
Probabilities: 0 (18.20%) | 1 (39.92%) | 2 (41.88%)


## Run this so that the training autogenerates a plot after finishing


In [19]:
import datetime
import matplotlib.pyplot as plt

def plot_lists_with_colors(list1, list2, label1="List 1", label2="List 2", color1="blue", color2="red"):
    # Ensure lists have equal length (or adjust as needed)
    min_len = min(len(list1), len(list2))
    x_values = np.arange(min_len)

    # Create the plot
    plt.figure(figsize=(10, 6))  # Adjust figure size if needed

    # Plot List 1
    plt.plot(x_values, list1[:min_len], color=color1, label=label1, linestyle='none', marker='o')
    plt.plot(x_values, list2[:min_len], color=color2, label=label2, linestyle='none', marker='o')
    current_time = datetime.datetime.now()
    
    # Add Labels and Title
    plt.xlabel("Epoch") 
    plt.ylabel("Accuracy (%)")
    plt.title(f"Bert - Midwest data ({current_time.strftime("%Y-%m-%d %H:%m")})")

    plt.legend()
    plt.grid(True)

    file_name = f'./sshots/model-{current_time.strftime("%Y-%m-%d--%H%m")}.png'
    plt.savefig(file_name)
    
    # Show the Plot
    plt.show()

## Training the BERT model begins here

In [20]:
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split


# Create dataset from data
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length #store max length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        
        inputs = self.tokenizer(
            text,
            padding="max_length", #important
            truncation=True, #important
            max_length=self.max_length, #important
            return_tensors="pt")
        
        input_ids = inputs['input_ids'].flatten()
        attention_mask = inputs['attention_mask'].flatten()
        label_tensor = torch.tensor(label)

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': label_tensor
        }

In [21]:
from sklearn.utils import class_weight

# Ratings are not evenly distributed, this creates class weights
def calculate_class_weights(labels):
    class_weights = class_weight.compute_class_weight('balanced', classes=np.unique(labels), y=labels)
    return torch.tensor(class_weights, dtype=torch.float)

In [95]:
# Export the text and ratings to a list, this part is necessary otherwise pandas keeps the index
X_train = df_train['text'].values.tolist()
y_train = df_train['pnn'].values.tolist()
X_test = df_test['text'].values.tolist()
y_test = df_test['pnn'].values.tolist()

# The batch size seems to need to be pretty small, anything more than 32 crashed my 3070 with 8GB VRAM
# monitor RAM use in the terminal with "watch -n5 nvidia-smi"
batch_size = 8
max_length = 128
learning_rate = 2e-5
weight_decay = .01

## FREEZE the base BERT parameters and only train the classification layer
for param in model.roberta.parameters():
    param.requires_grad = False

train_dataset = SentimentDataset(X_train, y_train, tokenizer, max_length=max_length)
test_dataset = SentimentDataset(X_test, y_test, tokenizer, max_length=max_length)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size)

# Adjust learning rate and weight decay
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

# Calculate class weights only after the split
class_weights = calculate_class_weights(y_train).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

print('Class weights:', class_weights)

Class weights: tensor([4.3860, 3.9216, 0.3973])


In [99]:
def test_cycle(model, dataloader):
    # Put in evaluation mode
    print('Testing...', end='\r')
    model.eval()
    
    all_predicted_labels, all_true_labels = [], []
    
    with torch.no_grad():
        for batch in test_dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            outputs = model(input_ids, attention_mask=attention_mask)
            
            _, predicted_labels = torch.max(outputs.logits, dim=1)
            all_predicted_labels.extend(predicted_labels.cpu().numpy())
            all_true_labels.extend(labels.cpu().numpy())

    ## Accuracy and F1 used to determine early exit
    acc = accuracy_score(all_true_labels, all_predicted_labels)
    f1 = f1_score(all_true_labels, all_predicted_labels, average='weighted')
    
    return acc, f1

In [111]:
def train_roberta_early_stopping(model, train_dataloader, test_dataloader, optimizer, epochs=100, patience=5, device='cuda'):
    print(f'Trianing on device ({device}) with {epochs} epochs')
    
    # Keep track of some data
    accs = [[], []] #0 = Train, 1 = Test
    losses = []
    f1s = []
    
    num_training_steps = len(train_dataloader) * epochs

    # Learning rate scheduler
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

    # Set best value as low as possible
    best_val_f1 = -np.inf
    patience_counter = 0

    for epoch in range(epochs):
        
        model.train()
    
        train_loss = 0
        all_predicted_labels, all_true_labels = [], []
    
        for batch in tqdm(train_dataloader, desc=f"Epoch {epoch + 1}"):
            optimizer.zero_grad()
    
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
    
            ## used for calculating accuracy
            _, predicted_labels = torch.max(outputs.logits, dim=1)
            all_predicted_labels.extend(predicted_labels.cpu().numpy())
            all_true_labels.extend(labels.cpu().numpy())
    
            ## Training
            loss = outputs.loss
            train_loss += loss.item()
            
            loss.backward()
            
            optimizer.step()
            scheduler.step()

        # Keep track of learning acc and loss
        acc = accuracy_score(all_true_labels, all_predicted_labels)
        trin_loss = train_loss / len(train_dataloader)
        
        accs[0].append(100*acc)
        losses.append(loss)
        
        # Test cycle
        acc, val_f1 = test_cycle(model=model, dataloader=test_dataloader)
        
        accs[1].append(100*acc)
        f1s.append(val_f1)

        # Keep track of progress
        print(f'Train loss: {loss:.5f}, acc {accs[0][epoch]:.2f}% | Test acc: {accs[1][epoch]:.2f}%, f1 = {val_f1:.5f}')
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            patience_counter = 0
            torch.save(model.state_dict(), 'best_roberta_model.pth') #save best model
            
        else:
            patience_counter += 1
            if patience_counter >= patience:
                
                print('Early stopping triggered')
                plot_lists_with_colors(accs[0], accs[1], label1="Train accuracy", label2="Test accuracy")
                break #exit training loop.
                
    model.load_state_dict(torch.load('best_roberta_model.pth')) #load best model
    plot_lists_with_colors(accs[0], accs[1], label1="Train accuracy", label2="Test accuracy")
    
    return model

In [113]:
# Train the model here
model = train_roberta_early_stopping(model=model, 
                                     train_dataloader=train_dataloader, 
                                     test_dataloader=test_dataloader, 
                                     optimizer=optimizer, 
                                     epochs=100, 
                                     patience=5,
                                     device=device)

Trianing on device (cpu) with 100 epochs


Epoch 1: 100%|████████████████████████████████| 125/125 [00:58<00:00,  2.12it/s]


Train loss: 0.18882, acc 87.80% | Test acc: 88.30%, f1 = 0.8667739379492571


Epoch 2: 100%|████████████████████████████████| 125/125 [01:01<00:00,  2.03it/s]


Train loss: 0.02146, acc 87.70% | Test acc: 88.60%, f1 = 0.8697575396047382


...

Train loss: 0.57992, acc 89.20% | Test acc: 88.00%, f1 = 0.8566320236955897


Epoch 4: 100%|████████████████████████████████| 125/125 [01:05<00:00,  1.92it/s]


Train loss: 0.35358, acc 88.90% | Test acc: 88.60%, f1 = 0.8729769137583498


Epoch 5:  15%|█████                            | 19/125 [00:10<00:56,  1.88it/s]


KeyboardInterrupt: 

## Suggestions

 Set aside cross-validation set - 
 
 Total random seed pytorch level torch.manual_seed() - DONE
  
 Early exit with high number of epochs
 
 Play with groupings of ratings (1--5, pos/neg only -- create neutral based on logits) - TRIED WITH 0--4

 Learning rates: 5e-5, 4e-5, 3e-5, and 2e-5, .0001. Learning rate scheduler

 Investigate BERT classifier more

 Try different BERT model -- roberta? https://huggingface.co/AnkitAI/reviews-roberta-base-sentiment-analysis